# Voice Assistant

A speech pipeline that chains Sarvam speech-to-text, sarvam-m chat, and Sarvam
text-to-speech into a single voice assistant loop.

## Pipeline

```
Audio input (WAV)
        |
        v
transcribe_audio()      <- Sarvam saarika:v2  (POST /speech-to-text)
        |
        v
generate_response()     <- Sarvam sarvam-m   (chat.completions)
        |
        v
synthesize_speech()     <- Sarvam bulbul:v1  (POST /text-to-speech)
        |
        v
outputs/response.wav
```

In [ ]:
%pip install sarvamai>=0.1.26 requests>=2.31.0 python-dotenv>=1.0.0

## Setup and API Key

Load environment variables and initialise the Sarvam AI client.
Set `SARVAM_API_KEY` in a `.env` file copied from `.env.example`.

In [ ]:
from __future__ import annotations

import base64
import math
import os
import struct
import wave
from pathlib import Path
from typing import Any

import requests
from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

STT_URL = "https://api.sarvam.ai/speech-to-text"
TTS_URL = "https://api.sarvam.ai/text-to-speech"

print("Setup complete.")

## Generate Sample Audio

Create a 1-second 440 Hz sine-wave WAV file for end-to-end testing.
Replace `sample_data/input.wav` with real speech audio for meaningful STT results.

In [ ]:
def generate_sample_audio(
    output_path: Path,
    duration_sec: float = 1.0,
    sample_rate: int = 16000,
    frequency: float = 440.0,
) -> Path:
    """Write a mono 16-bit sine-wave WAV to output_path.

    Args:
        output_path: Destination file path.
        duration_sec: Length of the tone in seconds.
        sample_rate: PCM sample rate in Hz.
        frequency: Sine-wave frequency in Hz.

    Returns:
        Path to the written WAV file.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    num_frames = int(sample_rate * duration_sec)

    with wave.open(str(output_path), "w") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)  # 16-bit PCM
        wf.setframerate(sample_rate)
        frames = struct.pack(
            f"<{num_frames}h",
            *(
                int(32767 * math.sin(2 * math.pi * frequency * i / sample_rate))
                for i in range(num_frames)
            ),
        )
        wf.writeframes(frames)

    return output_path


SAMPLE_AUDIO = generate_sample_audio(Path("sample_data/input.wav"))
print(f"Sample audio written: {SAMPLE_AUDIO}  ({SAMPLE_AUDIO.stat().st_size} bytes)")

## Speech-to-Text

`transcribe_audio` sends a WAV file to Sarvam `saarika:v2` and returns the transcript string.

In [ ]:
def transcribe_audio(
    audio_path: Path,
    api_key: str = SARVAM_API_KEY,
    language_code: str = "en-IN",
) -> str:
    """Transcribe a WAV file using Sarvam saarika:v2 STT.

    Args:
        audio_path: Path to the input WAV file.
        api_key: Sarvam API subscription key.
        language_code: BCP-47 language code for transcription (e.g. en-IN, hi-IN).

    Returns:
        Transcript string; empty string if the audio contains no recognisable speech.

    Raises:
        requests.HTTPError: On non-2xx API responses.
    """
    with audio_path.open("rb") as audio_file:
        response = requests.post(
            STT_URL,
            headers={"api-subscription-key": api_key, "Accept": "application/json"},
            files={"file": (audio_path.name, audio_file, "audio/wav")},
            data={"language_code": language_code, "model": "saarika:v2", "with_timestamps": "false"},
            timeout=30,
        )
    response.raise_for_status()
    return response.json().get("transcript", "")

### Transcription Demo

Run the STT function on the sample audio.
The synthetic tone will produce an empty transcript -- replace the file with real speech to see results.

In [ ]:
transcript = transcribe_audio(SAMPLE_AUDIO)
print(f"Transcript: {transcript!r}")
print("(Empty transcript is expected for a synthetic sine tone.)")

## Chat Completion

`generate_response` sends the transcript to Sarvam `sarvam-m` and returns the assistant reply.

In [ ]:
def generate_response(
    transcript: str,
    sarvam_client: SarvamAI = client,
) -> str:
    """Generate a conversational reply using Sarvam sarvam-m.

    Args:
        transcript: User utterance from STT.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Assistant reply string.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful voice assistant. "
                "Keep responses concise and conversational."
            ),
        },
        {"role": "user", "content": transcript},
    ]
    response = sarvam_client.chat.completions(messages=messages, model="sarvam-m")
    return response.choices[0].message.content

## Text-to-Speech

`synthesize_speech` sends the assistant reply to Sarvam `bulbul:v1` and saves the audio to
`outputs/response.wav`.

In [ ]:
def synthesize_speech(
    text: str,
    output_path: Path,
    api_key: str = SARVAM_API_KEY,
    language_code: str = "en-IN",
    speaker: str = "meera",
) -> Path:
    """Convert text to speech using Sarvam bulbul:v1 TTS and save as WAV.

    Args:
        text: Text to synthesise.
        output_path: Destination path for the output WAV file.
        api_key: Sarvam API subscription key.
        language_code: BCP-47 target language code.
        speaker: Sarvam TTS speaker identifier.

    Returns:
        Path to the written WAV file.

    Raises:
        requests.HTTPError: On non-2xx API responses.
    """
    response = requests.post(
        TTS_URL,
        headers={
            "api-subscription-key": api_key,
            "Accept": "application/json",
            "Content-Type": "application/json",
        },
        json={
            "inputs": [text],
            "target_language_code": language_code,
            "speaker": speaker,
            "pace": 1.0,
            "speech_sample_rate": 22050,
            "model": "bulbul:v1",
        },
        timeout=30,
    )
    response.raise_for_status()
    audio_b64 = response.json()["audios"][0]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(base64.b64decode(audio_b64))
    return output_path

## Full Voice Assistant Pipeline

`run_voice_pipeline` chains all three stages end-to-end.

In [ ]:
def run_voice_pipeline(
    audio_path: Path,
    output_path: Path = Path("outputs/response.wav"),
    sarvam_client: SarvamAI = client,
) -> dict[str, Any]:
    """Run the full Sarvam voice assistant pipeline.

    Steps:
        1. Transcribe input audio with saarika:v2 STT.
        2. Generate assistant reply with sarvam-m chat.
        3. Synthesise reply audio with bulbul:v1 TTS.

    Args:
        audio_path: Path to the input WAV file.
        output_path: Destination path for the synthesised response WAV.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Dict with keys: transcript, response, audio_path.
    """
    transcript = transcribe_audio(audio_path)
    response_text = generate_response(transcript, sarvam_client)
    audio_out = synthesize_speech(response_text, output_path)

    return {
        "transcript": transcript,
        "response": response_text,
        "audio_path": str(audio_out),
    }


# Demo
result = run_voice_pipeline(
    audio_path=SAMPLE_AUDIO,
    output_path=Path("outputs/response.wav"),
    sarvam_client=client,
)

print("Transcript :", result["transcript"])
print("Response   :", result["response"])
print("Audio saved:", result["audio_path"])

## Error Reference and Resources

### Common Errors

| Error | Cause | Fix |
|:------|:------|:----|
| `RuntimeError: SARVAM_API_KEY is not set` | Missing `.env` | Copy `.env.example` to `.env` and add your key |
| `requests.HTTPError: 401` | Invalid API key | Verify key at dashboard.sarvam.ai |
| `requests.HTTPError: 429` | Rate limit exceeded | Add a short delay between requests |
| `KeyError: transcript` | Unexpected STT response | Check audio is WAV format and 16 kHz sample rate |
| `KeyError: audios` | Unexpected TTS response | Ensure `inputs` is a non-empty list of non-empty strings |

### Resources

- [Sarvam AI Documentation](https://docs.sarvam.ai/)
- [Sarvam API Dashboard](https://dashboard.sarvam.ai/)
- [Sarvam STT API reference](https://docs.sarvam.ai/api-reference-docs/speech-to-text/transcribe)
- [Sarvam TTS API reference](https://docs.sarvam.ai/api-reference-docs/text-to-speech/convert)